In [1]:
import os
import re
import sqlite3

DUMP = "../../../data/northwind.sql"
DB = "northwind.sqlite"

with open(DUMP) as f:
    sql = f.read()

sql = re.sub(r"^SET .*?;\s*$", "", sql, flags=re.M)
sql = re.sub(r"ALTER TABLE ONLY[^;]*;", "", sql, flags=re.S)
sql = sql.replace("bytea", "BLOB")
sql = sql.replace(r"'\x'", "NULL")

if os.path.exists(DB):
    os.remove(DB)
conn = sqlite3.connect(DB)
conn.executescript(sql)
conn.commit()

cur = conn.cursor()
for table in ["customers", "orders", "order_details", "products", "employees"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]}")

customers: 91
orders: 830
order_details: 2155
products: 77
employees: 9



# Exercices sur les jointures

## Exercice 1 : Left join 
- Rapatrier tous les order_id de la table orders et les customer_id correspondants, même si certains order_id n'ont pas de customer_id correspondant

## Exercice 2 : Inner join
- Rapatrier tous les order_id et customer_id de la table orders où il existe une correspondance entre les deux

## Exercice 3 : Full outer join
- Combiner toutes les lignes de orders et customers en utilisant un FULL OUTER JOIN sur customer_id, y compris les lignes sans correspondance

## Exercice 4 : self join
- Utiliser un self join sur la table customers pour trouver les paires de clients de la même ville

# Exercices sur les agrégations avec GROUP BY
## Exercice 5 : group by basique
- Grouper les commandes de la table orders par ship_country et compter le nombre de commandes pour chaque pays

## Exercice 6 : group by avec fonction SUM()
- Calculer le fret total (freight) par ship_country dans la table orders

## Exercice 7 : group by avec join
- Joindre orders et customers et calculer le nombre total de commandes pour chaque company_name

## Exercice 8 : group by avec plusieurs colonnes
- Grouper les commandes par ship_city et ship_country dans orders, et calculer le fret total pour chaque groupe

# Plusieurs clauses

## Exercice 9 : 
- Grouper orders par ship_country et sélectionner les pays dont le fret total dépasse 1000 

# Exercice 1 : Left join
Rapatrier tous les order_id de la table orders et les customer_id correspondants, même si certains order_id n'ont pas de customer_id correspondant

In [7]:
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

cur.execute("""
    SELECT o.order_id, o.customer_id
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    LIMIT 10
""")

for row in cur:
    print(row)

conn.close()

(10248, 'VINET')
(10249, 'TOMSP')
(10250, 'HANAR')
(10251, 'VICTE')
(10252, 'SUPRD')
(10253, 'HANAR')
(10254, 'CHOPS')
(10255, 'RICSU')
(10256, 'WELLI')
(10257, 'HILAA')


# Exercice 2 : Inner join
Rapatrier tous les order_id et customer_id de la table orders où il existe une correspondance entre les deux

In [10]:
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

cur.execute("""
    SELECT o.order_id, o.customer_id
    FROM orders o
    INNER JOIN customers c ON o.customer_id = c.customer_id
    LIMIT 10
""")

for row in cur:
    print(row)

conn.close()

(10248, 'VINET')
(10249, 'TOMSP')
(10250, 'HANAR')
(10251, 'VICTE')
(10252, 'SUPRD')
(10253, 'HANAR')
(10254, 'CHOPS')
(10255, 'RICSU')
(10256, 'WELLI')
(10257, 'HILAA')


## Exercice 3 : Full outer join
- Combiner toutes les lignes de orders et customers en utilisant un FULL OUTER JOIN sur customer_id, y compris les lignes sans correspondance

In [13]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

cur.execute("""

SELECT c.customer_id, o.order_id
FROM customers c
FULL OUTER JOIN orders o ON c.customer_id = o.customer_id
LIMIT 10

""")

for row in cur:
    print(row)

conn.close()

('ALFKI', 10643)
('ALFKI', 10692)
('ALFKI', 10702)
('ALFKI', 10835)
('ALFKI', 10952)
('ALFKI', 11011)
('ANATR', 10308)
('ANATR', 10625)
('ANATR', 10759)
('ANATR', 10926)


## Exercice 4 : self join
- Utiliser un self join sur la table customers pour trouver les paires de clients de la même ville

In [22]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

cur.execute("""

SELECT 
  c1.company_name AS Customers1, 
  c2.company_name AS Customers2,
  c1.city
FROM 
  customers c1
INNER JOIN customers c2
  ON c1.city = c2.city AND c1.customer_id < c2.customer_id 
  ORDER BY c1.city
  LIMIT 10;
""")

for row in cur:
    print(row)

conn.close()

('Cactus Comidas para llevar', 'OcÃ©ano AtlÃ¡ntico Ltda.', 'Buenos Aires')
('Cactus Comidas para llevar', 'Rancho grande', 'Buenos Aires')
('OcÃ©ano AtlÃ¡ntico Ltda.', 'Rancho grande', 'Buenos Aires')
('Furia Bacalhau e Frutos do Mar', 'Princesa Isabel Vinhos', 'Lisboa')
('Around the Horn', "B's Beverages", 'London')
('Around the Horn', 'Consolidated Holdings', 'London')
('Around the Horn', 'Eastern Connection', 'London')
('Around the Horn', 'North/South', 'London')
('Around the Horn', 'Seven Seas Imports', 'London')
("B's Beverages", 'Consolidated Holdings', 'London')


# Exercices sur les agrégations avec GROUP BY
## Exercice 5 : group by basique
- Grouper les commandes de la table orders par ship_country et compter le nombre de commandes pour chaque pays

In [24]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

cur.execute("""
    SELECT ship_country, COUNT(order_id)
    FROM orders 
    GROUP BY ship_country
    ORDER BY COUNT(order_id) DESC


""")

for row in cur:
    print(row)

conn.close()

('USA', 122)
('Germany', 122)
('Brazil', 83)
('France', 77)
('UK', 56)
('Venezuela', 46)
('Austria', 40)
('Sweden', 37)
('Canada', 30)
('Mexico', 28)
('Italy', 28)
('Spain', 23)
('Finland', 22)
('Ireland', 19)
('Belgium', 19)
('Switzerland', 18)
('Denmark', 18)
('Argentina', 16)
('Portugal', 13)
('Poland', 7)
('Norway', 6)


## Exercice 6 : group by avec fonction SUM()
- Calculer le fret total (freight) par ship_country dans la table orders

In [26]:
# 1. Connexion à la base de données
conn = sqlite3.connect("northwind.sqlite")
cur = conn.cursor()

cur.execute("""
    SELECT ship_country , SUM(freight)
    FROM orders 
    GROUP BY ship_country
   

""")

for row in cur:
    print(row)

conn.close()

('Argentina', 598.580000293)
('Austria', 7391.50002476)
('Belgium', 1280.139979792)
('Brazil', 4880.190048354)
('Canada', 2198.089989098)
('Denmark', 1396.18999477)
('Finland', 910.890000137)
('France', 4237.8400108826)
('Germany', 11283.280008102)
('Ireland', 2755.239951)
('Italy', 864.440001709)
('Mexico', 1122.779989306)
('Norway', 275.49999809)
('Poland', 175.74000313)
('Portugal', 643.53000062)
('Spain', 861.89000205)
('Sweden', 3237.60000705)
('Switzerland', 1368.52999952)
('UK', 2954.269983626)
('USA', 13771.290043656)
('Venezuela', 2735.180008154)


## Exercice 7 : group by avec join
- Joindre orders et customers et calculer le nombre total de commandes pour chaque company_name

## Exercice 8 : group by avec plusieurs colonnes
- Grouper les commandes par ship_city et ship_country dans orders, et calculer le fret total pour chaque groupe

# Plusieurs clauses

## Exercice 9 : 
- Grouper orders par ship_country et sélectionner les pays dont le fret total dépasse 1000 